# Online RL Thesis — Sprint 1 · §2 Sensor Envelope

**目的**：在不做任何算法改进时，刻画 sensor × difficulty 二维空间内 vanilla SAC 的能力边界，作为后续 §3 privileged-critic 改进的 baseline 基线。

**对应计划**：[`docs/online_rl_thesis_plan.md`](../docs/online_rl_thesis_plan.md) §4.1。

**Pre-flight 前置**：必须先完成 [`sac_thesis_s0_preflight.ipynb`](sac_thesis_s0_preflight.ipynb) 的 P1 + P2，并把 u15 预算决策落到 thesis plan §10。

**实验矩阵**：

|              | u10_upstream | u15_upstream |
|---           |---           |---           |
| s0_k4        | 3 seed       | 3 seed       |
| s1_k4        | 3 seed       | 3 seed       |
| s2_k4        | 3 seed       | 3 seed       |

- 共 18 run，全部 vanilla SAC（**no LayerNorm / no UTD / no asym**）
- seeds: `46, 47, 50`
- objective: `efficiency_v2`
- 预期产出：3×2 heatmap（success rate / variance / path_efficiency 各一张）

**预期结论**：
- vanilla s0 × u15 显著低于 vanilla {s1, s2} × u15 → 这是 §3 要 attack 的 headroom
- 如果 §2 显示 vanilla s0 × u15 已 saturate → 重选难度（升到 u20 或类似）


## 0. 环境检查


In [ ]:
!nvidia-smi


In [ ]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 1. 挂载 Drive + cd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd


## 2. 通用 env 配置

`FLOW_PATH / TASK_GEOMETRY / TARGET_SPEED` 由 `BENCHMARK_KEY` 通过 `resolve_benchmark_protocol` 自动推导，所以这里不再设全局值（u10 与 u15 列在各自 cell 内 resolve）。

`U15_TOTAL_STEPS` 来自 Sprint 0 P1 的标定结果，请把数字改为实测值。


In [ ]:
import os

# ---- 与 §2 全局共享的协议（与 A0 保持一致，确保跨阶段可比） ----
os.environ['PYTHON_BIN']       = 'python3'
os.environ['DEVICE']           = 'cuda'
os.environ['STUDY_ROOT']       = 'online_thesis_v1'
os.environ['NUM_ENVS']         = '12'             # P0b 决策（plan §10.2）：12 比 6 快 1.45×
os.environ['SEEDS']            = '46 47 50'
os.environ['PROBES']           = 's0 s1 s2'
os.environ['HISTORY_LENGTH']   = '4'
os.environ['OBJECTIVE']        = 'efficiency_v2'
os.environ['ALGO_TAG']         = 'sac_vanilla'
os.environ['EVAL_EVERY']       = '10000'
os.environ['EVAL_EPISODES']    = '30'
os.environ['CHECKPOINT_EVERY'] = '50000'          # 比默认稀，省 Drive 空间；不影响 best/final/latest 三件套

# 注意：FLOW_PATH / TASK_GEOMETRY / TARGET_SPEED 不在此处设全局值；
# 它们随 BENCHMARK_KEY 自动 resolve（见下方 u10 / u15 cell）。

# ---- 来自 Sprint 0 P1 的标定（修改下面这行为实测值） ----
U10_TOTAL_STEPS = 600_000
U15_TOTAL_STEPS = 600_000   # ← 改为 P1 标定的预算（600k / 800k / 1_000_000）

print(f'NUM_ENVS        = {os.environ["NUM_ENVS"]}')
print(f'u10 total_steps = {U10_TOTAL_STEPS}')
print(f'u15 total_steps = {U15_TOTAL_STEPS}')


## 3. u10_upstream 列（3 sensor × 3 seed = 9 run）

`run_protocol_stage_common.sh` 内置 `[skip] ${RUN_DIR}` 重入逻辑——session 中断后重跑此 cell 会跳过已完成的 run。


In [ ]:
import os

from scripts.benchmark_catalog import resolve_benchmark_protocol
from scripts.notebook_utils import run_streaming

BENCHMARK_KEY_U10 = 'single_u10_upstream_tgt15'
proto_u10 = resolve_benchmark_protocol(BENCHMARK_KEY_U10)

env_u10 = {
    **os.environ,
    'STAGE_LABEL':   'S2_u10',
    'STAGE_DIR':     f'sec2_sensor_envelope/{BENCHMARK_KEY_U10}',
    'BENCHMARK_KEY': BENCHMARK_KEY_U10,
    'FLOW_PATH':     proto_u10['flow_path'],       # auto: U=1.0 wake (Re150)
    'TASK_GEOMETRY': proto_u10['task_geometry'],
    'TARGET_SPEED':  proto_u10['target_speed'],
    'TOTAL_STEPS':   str(U10_TOTAL_STEPS),
}

n_runs = len(env_u10['PROBES'].split()) * len(env_u10['SEEDS'].split())
print(f'[§2 u10] FLOW_PATH = {proto_u10["flow_path"]}')
print(f'[§2 u10] launching {n_runs} runs ...', flush=True)
# run_streaming → 每个 train_sac 子进程的进度按行实时刷出（PYTHONUNBUFFERED=1）
run_streaming(['bash', 'scripts/run_protocol_stage_common.sh'], env=env_u10)
print('[§2 u10] done')


## 4. u15_upstream 列（3 sensor × 3 seed = 9 run）


In [ ]:
BENCHMARK_KEY_U15 = 'single_u15_upstream_tgt15'
proto_u15 = resolve_benchmark_protocol(BENCHMARK_KEY_U15)

env_u15 = {
    **os.environ,
    'STAGE_LABEL':   'S2_u15',
    'STAGE_DIR':     f'sec2_sensor_envelope/{BENCHMARK_KEY_U15}',
    'BENCHMARK_KEY': BENCHMARK_KEY_U15,
    'FLOW_PATH':     proto_u15['flow_path'],       # auto: U=1.5 wake (Re250)
    'TASK_GEOMETRY': proto_u15['task_geometry'],
    'TARGET_SPEED':  proto_u15['target_speed'],
    'TOTAL_STEPS':   str(U15_TOTAL_STEPS),
}

n_runs = len(env_u15['PROBES'].split()) * len(env_u15['SEEDS'].split())
print(f'[§2 u15] FLOW_PATH = {proto_u15["flow_path"]}')
print(f'[§2 u15] launching {n_runs} runs ...', flush=True)
run_streaming(['bash', 'scripts/run_protocol_stage_common.sh'], env=env_u15)
print('[§2 u15] done')


## 5. 汇总：每个 (sensor, difficulty) 单元格的 mean/std


In [ ]:
import os

from scripts.notebook_utils import run_streaming

for stage_dir, bench in [
    ('sec2_sensor_envelope/single_u10_upstream_tgt15', 'single_u10_upstream_tgt15'),
    ('sec2_sensor_envelope/single_u15_upstream_tgt15', 'single_u15_upstream_tgt15'),
]:
    env_sum = {
        **os.environ,
        'STAGE_LABEL':   f'S2_{bench}',
        'STAGE_DIR':     stage_dir,
        'BENCHMARK_KEY': bench,
    }
    run_streaming(['bash', 'scripts/summarize_protocol_stage_common.sh'], env=env_sum)


## 6. 二维 heatmap 可视化

读两份 `summary/ablation_summary.csv`，拼成 sensor × difficulty 的矩阵图（success / variance / path_efficiency）。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('experiments/online_thesis_v1/sec2_sensor_envelope')
DIFFS = ['single_u10_upstream_tgt15', 'single_u15_upstream_tgt15']
SENSORS = ['s0_k4', 's1_k4', 's2_k4']

def load(diff):
    p = ROOT / diff / 'efficiency_v2' / 'sac_vanilla' / 'summary' / 'ablation_summary.csv'
    return pd.read_csv(p)

frames = {d: load(d) for d in DIFFS}
print({d: list(f.columns) for d, f in frames.items()})

def matrix(metric_mean: str, metric_std: str | None = None):
    M = np.full((len(SENSORS), len(DIFFS)), np.nan)
    S = np.full((len(SENSORS), len(DIFFS)), np.nan) if metric_std else None
    for j, d in enumerate(DIFFS):
        df = frames[d].set_index('method')
        for i, s in enumerate(SENSORS):
            if s in df.index:
                M[i, j] = df.loc[s, metric_mean]
                if metric_std:
                    S[i, j] = df.loc[s, metric_std]
    return M, S

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))

# 1) success rate mean
M, _ = matrix('eval_success_rate_mean')
im = ax[0].imshow(M, cmap='viridis', aspect='auto', vmin=0, vmax=1)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax[0].text(j, i, f'{M[i,j]:.2f}', ha='center', va='center',
                   color='white' if M[i,j] < 0.6 else 'black', fontsize=11)
ax[0].set_xticks(range(len(DIFFS))); ax[0].set_xticklabels(['u10_up', 'u15_up'])
ax[0].set_yticks(range(len(SENSORS))); ax[0].set_yticklabels(SENSORS)
ax[0].set_title('success_rate (mean over 3 seeds)'); plt.colorbar(im, ax=ax[0])

# 2) success rate std
_, S = matrix('eval_success_rate_mean', 'eval_success_rate_std')
im = ax[1].imshow(S, cmap='Reds', aspect='auto', vmin=0, vmax=0.3)
for i in range(S.shape[0]):
    for j in range(S.shape[1]):
        ax[1].text(j, i, f'{S[i,j]:.2f}', ha='center', va='center', fontsize=11)
ax[1].set_xticks(range(len(DIFFS))); ax[1].set_xticklabels(['u10_up', 'u15_up'])
ax[1].set_yticks(range(len(SENSORS))); ax[1].set_yticklabels(SENSORS)
ax[1].set_title('success_rate (std over 3 seeds)'); plt.colorbar(im, ax=ax[1])

# 3) path_efficiency mean (success-conditioned)
M, _ = matrix('eval_path_efficiency_mean')
im = ax[2].imshow(M, cmap='cividis', aspect='auto')
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax[2].text(j, i, f'{M[i,j]:.2f}', ha='center', va='center',
                   color='white' if M[i,j] < np.nanmean(M) else 'black', fontsize=11)
ax[2].set_xticks(range(len(DIFFS))); ax[2].set_xticklabels(['u10_up', 'u15_up'])
ax[2].set_yticks(range(len(SENSORS))); ax[2].set_yticklabels(SENSORS)
ax[2].set_title('path_efficiency (mean)'); plt.colorbar(im, ax=ax[2])

fig.suptitle('§2 Sensor Envelope on Vanilla SAC', fontsize=13, y=1.02)
fig.tight_layout(); plt.show()


## 7. 决策记录 — §3 是否启动？

阅读上述 heatmap，回答以下问题，并把结论写入 [`docs/online_rl_thesis_report.md`](../docs/online_rl_thesis_report.md) 的 §2 章节：

| 问题 | 期望答案 | 实测 |
|---|---|---|
| `vanilla s0 × u15_upstream` 是否显著低于 `vanilla {s1, s2} × u15_upstream`？ | 是（gap > 1σ） | ___ |
| s0 × u10_upstream 是否也明显落后？还是已 saturate？ | 中等表现 | ___ |
| s2 是否 seed 方差仍然偏大（如 A0 那样）？ | 可能 | ___ |
| 整体最弱单元格是否 = `s0 × u15_upstream`？ | 是 | ___ |

**触发 §3 的硬条件**：`s0 × u15_upstream` 必须有 ≥ 0.2 的 success-rate gap 相对 `{s1, s2} × u15_upstream`，否则 privileged-critic 在饱和点上没有 headroom 可改。

如果 §3 触发：进 [`sac_thesis_s3_privileged_critic.ipynb`](sac_thesis_s3_privileged_critic.ipynb)（待创建）。
